# Milestone 1: Automated Pipeline for arXiv Metadata, TeX Sources, and References

Notebook này phục vụ việc thực thi trọn vẹn toàn bộ quy trình Milestone 1 cho từng bài báo trong khoảng `start_id` đến `end_id`:
1. **Cào Metadata** từ arXiv API -> lưu `data/papers.jsonl` và ghi thống kê **Bảng 1: `m1_discovery_stats.csv`**.
2. **Tải mã nguồn & giải nén** chỉ giữ lại các file `.tex` và `.bib` (loại bỏ hoàn toàn các file ảnh/hình nhị phân, `.sty`, `.cls`, `.pdf`) -> lưu vào `data/raw/<clean_arxiv_id>/` và ghi thống kê **Bảng 2: `m1_download_stats.csv`**.
3. **Cào danh sách trích dẫn (References)** từ Semantic Scholar Graph API -> lưu `data/references.jsonl` và ghi thống kê **Bảng 3: `m1_reference_stats.csv`**.
4. **Đo lường Benchmark Hệ thống** tự động trong suốt quá trình chạy -> ghi thống kê **Bảng 4: `m1_system_benchmark.csv`**.

### Cơ chế chống lỗi mạng & quá tải (Rate limit & Retry):
- Tuân thủ khoảng cách tối thiểu giữa các request: $\ge 3.0$s với arXiv, $\ge 1.0$s với Semantic Scholar.
- Tự động retry với exponential backoff khi gặp lỗi HTTP 429 (Too Many Requests), HTTP 500/502/503/504, và sự cố gián đoạn kết nối mạng.
- Fail fast không retry với HTTP 404.
- Hỗ trợ Resume: tự động nhận diện và bỏ qua các ID đã hoàn thành khi chạy lại.

## 1. Kết nối Google Drive và Thiết lập Môi trường (trên Colab)

In [ ]:
# Chạy cell này nếu bạn thực thi trên Google Colab
try:
    from google.colab import drive
    import os

    drive.mount('/content/drive')

    # Thư mục lưu trữ dự án trên Google Drive của bạn
    PROJECT_DIR = '/content/drive/MyDrive/Colab_Notebooks/arXiv-Project'

    if not os.path.exists(PROJECT_DIR):
        print(f"Thư mục {PROJECT_DIR} chưa tồn tại, đang clone từ GitHub...")
        !git clone -b cv1-metadata https://github.com/ahtuan171/arXiv-Project.git "{PROJECT_DIR}"
    else:
        print("Đang cập nhật mã nguồn mới nhất từ GitHub...")
        !git -C "{PROJECT_DIR}" pull origin cv1-metadata

    os.chdir(PROJECT_DIR)
    print("Thư mục làm việc hiện tại:", os.getcwd())
except ImportError:
    print("Đang chạy ở môi trường cục bộ (Local).")

In [ ]:
# Cài đặt các thư viện cần thiết
%pip install -r requirements.txt

## 2. Khai báo khoảng arXiv-ID cần xử lý

In [ ]:
# Gán giá trị vào 2 biến start_id và end_id
# Hỗ trợ cả định dạng chuỗi 'YYMM.NNNNN' hoặc số nguyên kèm tham số month
start_id = "2502.13235"
end_id = "2502.13245"

# Giới hạn số bài báo chạy thử nghiệm (đặt 0 để chạy toàn bộ dải ID)
LIMIT = 0

from src.utils import parse_arxiv_range
id_list = parse_arxiv_range(start_id, end_id)
if LIMIT > 0:
    id_list = id_list[:LIMIT]

print(f"-> Tổng số bài báo sẽ xử lý: {len(id_list)}")
print("ID đầu tiên:", id_list[0] if id_list else "None")
print("ID cuối cùng:", id_list[-1] if id_list else "None")

## 3. Khởi động Bộ theo dõi Benchmark Hệ thống (Bảng 4)

In [ ]:
from pathlib import Path
from src import config
from src.benchmark import SystemBenchmarkTracker

# Các thư mục lưu trữ theo đúng quy định:
# Nguồn giải nén lưu tại data/raw, thống kê CSV lưu tại data/csv
RAW_DIR = config.DATA_DIR / "raw"
CSV_DIR = config.DATA_DIR / "csv"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

# Khởi động background resource tracker (đo RAM, Disk và Wall Time)
tracker = SystemBenchmarkTracker(
    storage_dirs=[RAW_DIR, CSV_DIR, config.DATA_DIR],
    sample_interval_sec=0.5,
)
tracker.start()
print("-> Đã kích hoạt SystemBenchmarkTracker để theo dõi Bảng 4 (m1_system_benchmark.csv).")

## 4. Bước 1: Cào Metadata bài báo từ arXiv API (Bảng 1: `m1_discovery_stats`)

In [ ]:
from src.metadata_module import crawl_papers

print("=== BẮT ĐẦU CÀO METADATA BÀI BÁO (arXiv API) ===")
stats_papers = crawl_papers(
    id_list=id_list,
    out_path=config.DATA_DIR / "papers.jsonl",
    csv_path=CSV_DIR / "m1_discovery_stats.csv",
    resume=True,
)
print("-> Kết quả cào Metadata:", stats_papers)

## 5. Bước 2: Tải và Giải nén File Nguồn TeX (.tex và .bib ONLY) (Bảng 2: `m1_download_stats`)

In [ ]:
from src.download_module.downloader import download_papers_range

print("=== BẮT ĐẦU TẢI VÀ GIẢI NÉN MÃ NGUỒN (.tex, .bib ONLY) ===")
df_downloads = download_papers_range(
    start_id=id_list[0],
    end_id=id_list[-1],
    raw_dir=RAW_DIR,
    csv_dir=CSV_DIR,
    delay=config.ARXIV_MIN_INTERVAL,
    resume=True,
    show_progress=True,
)
print(f"-> Đã hoàn thành tải mã nguồn cho {len(df_downloads)} bài báo.")

## 6. Bước 3: Cào Danh sách Trích dẫn từ Semantic Scholar (Bảng 3: `m1_reference_stats`)

In [ ]:
from src.crawl_references_module import crawl_references_range

print("=== BẮT ĐẦU CÀO DANH SÁCH TRÍCH DẪN (Semantic Scholar API) ===")
df_refs = crawl_references_range(
    id_list=id_list,
    out_path=config.DATA_DIR / "references.jsonl",
    csv_path=CSV_DIR / "m1_reference_stats.csv",
    resume=True,
)
print(f"-> Đã hoàn thành cào trích dẫn cho {len(df_refs)} bài báo.")

## 7. Bước 4: Hoàn thành Benchmark và Lưu Bảng 4 (`m1_system_benchmark`)

In [ ]:
print("=== KẾT THÚC VÀ LƯU BENCHMARK HỆ THỐNG ===")
df_benchmark = tracker.stop(
    total_papers=len(id_list),
    csv_path=CSV_DIR / "m1_system_benchmark.csv",
)
print("-> Bảng 4: m1_system_benchmark:")
display(df_benchmark)

## 8. Kiểm tra và Trích xuất Toàn bộ 4 Bảng Thống kê Milestone 1

In [ ]:
import pandas as pd

print("=================================================================")
print("TỔNG HỢP CÁC BẢNG DỮ LIỆU MILESTONE 1 (LƯU TẠI data/csv/)")
print("=================================================================")

t1_file = CSV_DIR / "m1_discovery_stats.csv"
t2_file = CSV_DIR / "m1_download_stats.csv"
t3_file = CSV_DIR / "m1_reference_stats.csv"
t4_file = CSV_DIR / "m1_system_benchmark.csv"

if t1_file.exists():
    print("\n--- 1. BẢNG 1: m1_discovery_stats (Khám phá Metadata) ---")
    df1 = pd.read_csv(t1_file, dtype={"arxiv_id": str})
    display(df1.head(10))

if t2_file.exists():
    print("\n--- 2. BẢNG 2: m1_download_stats (Tải nguồn TeX / Bib) ---")
    df2 = pd.read_csv(t2_file, dtype={"arxiv_id": str})
    display(df2.head(10))

if t3_file.exists():
    print("\n--- 3. BẢNG 3: m1_reference_stats (Trích dẫn Semantic Scholar) ---")
    df3 = pd.read_csv(t3_file, dtype={"arxiv_id": str})
    display(df3.head(10))

if t4_file.exists():
    print("\n--- 4. BẢNG 4: m1_system_benchmark (Benchmark CPU/RAM/Disk trên Colab) ---")
    df4 = pd.read_csv(t4_file)
    display(df4)